In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("cuda available =", torch.cuda.is_available())
print("device count =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0 =", torch.cuda.get_device_name(0))

CUDA_VISIBLE_DEVICES = 1
LD_LIBRARY_PATH = None
cuda available = True
device count = 1
device 0 = NVIDIA RTX A6000


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import time
import uuid
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class HLCMHardNegGRPOConfig:
    # data
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    # outputs
    out_dir: str = "runs/hlcm_mawps_hardneg_grpo_v1"
    cache_dir: str = "mawps_cached_hardneg_v1"
    ref_logits_dir: str = "mawps_cached_ref_logits_v1"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 1

    # ranking setup
    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    # prompt
    instruction: str = (
        "Solve the math word problem.\n"
        "Return only the final numeric answer."
    )

    # optimization
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 2
    num_workers: int = 0
    max_grad_norm: float = 1.0
    weight_decay: float = 0.01
    use_bf16: bool = True

    # SFT stage
    sft_epochs: int = 12
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO stage
    run_grpo: bool = True
    grpo_epochs: int = 4
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03
    grpo_group_size: int = 8
    grpo_beta_kl: float = 0.02
    grpo_policy_temperature: float = 1.0
    mcq_logit_temperature: float = 0.1
    entropy_bonus: float = 0.001
    use_group_relative_advantage: bool = True

    # rewards
    reward_correct: float = 1.0
    reward_incorrect: float = 0.0

    # stability
    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    # checkpointing / disk
    save_last_every_epoch: bool = False
    max_prediction_files_to_keep: int = 2

    # misc
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def prune_prediction_files(out_dir: str, prefix: str, keep: int):
    files = []
    for fn in os.listdir(out_dir):
        if fn.startswith(prefix) and fn.endswith(".json"):
            full = os.path.join(out_dir, fn)
            files.append((os.path.getmtime(full), full))
    files.sort()
    if len(files) > keep:
        for _, path in files[:-keep]:
            try:
                os.remove(path)
            except Exception:
                pass


def assert_finite(name: str, x: torch.Tensor):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(f"{name} has non-finite values; bad_count={bad}; shape={tuple(x.shape)}")


def sanitize_tensor(x: torch.Tensor, clamp_value: float, replace_nonfinite_with_zero: bool) -> torch.Tensor:
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    x = torch.clamp(x, -clamp_value, clamp_value)
    return x


def load_normalizer(normalizer_path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        return None, None
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    torch.save(payload, tmp)
    os.replace(tmp, path)


# ============================================================
# NUMERIC UTILS
# ============================================================

def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None

    try:
        return float(s)
    except Exception:
        pass

    frac_match = re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s)
    if frac_match:
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None
    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""

    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")
    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        candidate = text.split("####")[-1].strip()
        return canonicalize_numeric_str(candidate)

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    all_candidates = frac_matches + dec_matches

    if all_candidates:
        best = None
        best_pos = -1
        for m in all_candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos
        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def numeric_equal(a: str, b: str, tol: float = 1e-6) -> bool:
    fa = safe_float_from_fraction_or_decimal(a)
    fb = safe_float_from_fraction_or_decimal(b)
    if fa is not None and fb is not None:
        return abs(fa - fb) <= tol
    return canonicalize_numeric_str(a) == canonicalize_numeric_str(b)


def try_parse_float(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    try:
        x = float(s)
        if math.isfinite(x):
            return float(x)
    except Exception:
        return None
    return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return (x is not None) and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    if "." not in s:
        return 0
    return len(s.split(".")[-1])


# ============================================================
# DATA
# ============================================================

def normalize_local_mawps_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    final_numeric_answer = extract_final_numeric_answer(answer)

    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = final_numeric_answer
    return row


def load_mawps_local(cfg: HLCMHardNegGRPOConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )
    return raw["train"], raw["validation"]


def build_mawps_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")

    n_last = max(1, min(n_last, len(layers)))
    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        summed = (hs * attn).sum(dim=1)
        denom = attn.sum(dim=1).clamp_min(1.0)
        out = summed / denom
        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# HARD NEGATIVE ANSWER POOL
# ============================================================

def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_mawps_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small to build negative candidates.")
    return out


def build_answer_pool_metadata(answer_pool: List[str]) -> List[Dict[str, Any]]:
    meta = []
    for ans in answer_pool:
        x = try_parse_float(ans)
        meta.append({
            "answer": ans,
            "value": x,
            "abs_value": abs(x) if x is not None else float("inf"),
            "is_integerish": is_integerish_str(ans),
            "decimals": num_decimal_places(ans),
            "str_len": len(ans),
        })
    return meta


def deterministic_fallback_order_key(candidate_answer: str, gold_answer: str, question: str) -> Tuple[int, str]:
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return (stable_int_hash(seed_text), candidate_answer)


def choose_hard_negative_answers(
    gold_answer: str,
    question: str,
    answer_pool_meta: List[Dict[str, Any]],
    k_neg: int,
    close_k: int = 64,
) -> List[str]:
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []
    for item in answer_pool_meta:
        cand = item["answer"]
        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        same_sign_flag = 1 if same_sign(gold_value, val) else 0
        same_int_flag = 1 if item["is_integerish"] == gold_is_int else 0
        same_dec_flag = 1 if item["decimals"] == gold_decimals else 0
        len_gap = abs(item["str_len"] - gold_len)
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append({
            "answer": cand,
            "dist": dist,
            "same_sign": same_sign_flag,
            "same_int": same_int_flag,
            "same_dec": same_dec_flag,
            "len_gap": len_gap,
            "abs_gap": abs_gap,
            "fallback_hash": fallback_hash,
        })

    candidates.sort(
        key=lambda z: (
            z["dist"],
            -z["same_sign"],
            -z["same_int"],
            -z["same_dec"],
            z["abs_gap"],
            z["len_gap"],
            z["fallback_hash"],
            z["answer"],
        )
    )

    close_candidates = candidates[:max(k_neg * 4, close_k)]
    if len(close_candidates) == 0:
        raise ValueError("No negative answers available.")

    selected = []
    seen = set()

    def add_if_new(ans: str):
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)

    for item in close_candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    for item in candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    return selected[:k_neg]


# ============================================================
# CACHE BUILD
# ============================================================

def cache_file_path(cfg: HLCMHardNegGRPOConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def ref_logits_file_path(cfg: HLCMHardNegGRPOConfig, split_name: str) -> str:
    ensure_dir(cfg.ref_logits_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.ref_logits_dir,
        f"{split_name}_{hard_tag}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def normalize_mawps_example_for_ranking(
    ex: Dict[str, Any],
    cfg: HLCMHardNegGRPOConfig,
    answer_pool_meta: List[Dict[str, Any]],
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_mawps_row(ex)
    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_mawps_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(
    cfg: HLCMHardNegGRPOConfig,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
    answer_pool_meta: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_mawps_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool_meta=answer_pool_meta,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)
            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
                "question_text": q_text,
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []
    question_texts = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]
        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])
        question_texts.append(item["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


# ============================================================
# HLCM BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: HLCMHardNegGRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: HLCMHardNegGRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# HLCM FORWARD
# ============================================================

def hlcm_encode_full_memory_safe(model: HyperbolicLCM, x: torch.Tensor) -> torch.Tensor:
    if not hasattr(model, "encode_inputs") or not hasattr(model, "layers"):
        return model(x)

    layers = list(model.layers)
    if len(layers) == 0:
        return model(x)

    first_trainable = len(layers)
    for i, layer in enumerate(layers):
        has_grad = any(p.requires_grad for p in layer.parameters())
        if has_grad:
            first_trainable = i
            break

    if first_trainable <= 0:
        h = model.encode_inputs(x)
        for layer in layers:
            h = layer(h)
        return h

    with torch.no_grad():
        h = model.encode_inputs(x)
        for layer in layers[:first_trainable]:
            h = layer(h)

    h = h.detach()
    for layer in layers[first_trainable:]:
        h = layer(h)

    return h


def hlcm_tangent_sequence(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if cfg.strict_finite_checks:
        assert_finite("input_x_before_norm", x)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h = hlcm_encode_full_memory_safe(model, x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    if cfg.strict_finite_checks:
        assert_finite("hlcm_h_tan", h_tan)

    return h_tan


def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
) -> torch.Tensor:
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)
    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask.to(h_tan.device)[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


# ============================================================
# LOGITS / LOSSES
# ============================================================

def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        ec = ec.reshape(B, (k1 - k0), -1)
        ec = F.normalize(ec, dim=-1)

        chunk_logits = torch.einsum("bd,bkd->bk", e_q, ec)
        logits_list.append(chunk_logits)

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
):
    logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm_detailed(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0, "chance_acc": 0.0}

    model.eval()
    tot_loss = 0.0
    tot_acc = 0.0
    tot_chance = 0.0
    n = 0

    for batch in loader:
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        labels = batch["label"].to(logits.device, non_blocking=True)
        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=1)
        bs = labels.size(0)
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        tot_loss += float(loss.item()) * bs
        tot_acc += float((preds == labels).float().sum().item())
        for k in choice_counts:
            tot_chance += 1.0 / max(1, int(k))
        n += bs

    return {
        "loss": tot_loss / max(1, n),
        "acc": tot_acc / max(1, n),
        "chance_acc": tot_chance / max(1, n),
    }


@torch.no_grad()
def dump_eval_predictions(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
    out_path: str,
):
    model.eval()
    rows = []

    for batch in loader:
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        probs = F.softmax(logits, dim=-1)
        pred_idx = logits.argmax(dim=1).detach().cpu().tolist()
        gold_idx = batch["label"].detach().cpu().tolist()
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        for i in range(len(pred_idx)):
            p = pred_idx[i]
            g = gold_idx[i]
            valid_k = int(choice_counts[i])
            logits_i = logits[i, :valid_k].detach().cpu()
            probs_i = probs[i, :valid_k].detach().cpu()
            choices = batch["choice_texts"][i]
            rows.append({
                "question_text": batch["question_texts"][i],
                "gold_answer": batch["gold_answers"][i],
                "pred_answer": choices[p],
                "gold_choice_index": g,
                "pred_choice_index": p,
                "correct": int(p == g),
                "choice_texts": choices,
                "choice_logits": [float(x) for x in logits_i.tolist()],
                "choice_probs": [float(x) for x in probs_i.tolist()],
            })

    safe_json_dump(rows, out_path)


# ============================================================
# OPTIMIZER
# ============================================================

def make_optimizer_and_scheduler(trainable_params, lr: float, total_steps: int, warmup_ratio: float, weight_decay: float):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError("No trainable parameters found.")

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGITS
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: HLCMHardNegGRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()
    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# SFT STAGE
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()))

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] val_loss={base_eval['loss']:.4f} val_acc={base_eval['acc']:.4f} chance={base_eval['chance_acc']:.4f}")
    append_dict_to_csv(eval_csv, {
        "epoch": 0,
        "train_loss": "",
        "val_loss": base_eval["loss"],
        "val_acc": base_eval["acc"],
        "val_chance_acc": base_eval["chance_acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    global_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)
        eval_metrics = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={eval_metrics['loss']:.4f} val_acc={eval_metrics['acc']:.4f} chance={eval_metrics['chance_acc']:.4f}"
        )

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })
        append_dict_to_csv(eval_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": eval_metrics["loss"],
            "val_acc": eval_metrics["acc"],
            "val_chance_acc": eval_metrics["chance_acc"],
        })

        if eval_metrics["acc"] > best_acc:
            best_acc = eval_metrics["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_val_acc": best_acc,
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        if cfg.save_last_every_epoch:
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "sft_last",
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "sft_last.pt"),
            )

        cuda_cleanup()

    model.load_state_dict(best_state, strict=True)
    return {"best_acc": best_acc, "best_state": best_state}


# ============================================================
# GRPO STAGE
# ============================================================

@torch.no_grad()
def sample_group_actions(logits: torch.Tensor, group_size: int, policy_temperature: float) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)
    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
    }
    return total_loss, stats


def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMHardNegGRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()))

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] val_loss={base_eval['loss']:.4f} val_acc={base_eval['acc']:.4f} chance={base_eval['chance_acc']:.4f}")
    append_dict_to_csv(eval_csv, {
        "epoch": 0,
        "train_loss": "",
        "val_loss": base_eval["loss"],
        "val_acc": base_eval["acc"],
        "val_chance_acc": base_eval["chance_acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    global_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        epoch_loss_sum = 0.0
        epoch_reward_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)
            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(model, batch, ref_logits_batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(model, batch, ref_logits_batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_reward_sum += float(stats["reward_mean"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                reward=f"{epoch_reward_sum / max(1, epoch_count):.4f}",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_reward = epoch_reward_sum / max(1, epoch_count)
        eval_metrics = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] train_loss={train_loss:.4f} train_reward={train_reward:.4f} "
            f"val_loss={eval_metrics['loss']:.4f} val_acc={eval_metrics['acc']:.4f} chance={eval_metrics['chance_acc']:.4f}"
        )

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reward": train_reward,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
        })
        append_dict_to_csv(eval_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reward": train_reward,
            "val_loss": eval_metrics["loss"],
            "val_acc": eval_metrics["acc"],
            "val_chance_acc": eval_metrics["chance_acc"],
        })

        if eval_metrics["acc"] > best_acc:
            best_acc = eval_metrics["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_val_acc": best_acc,
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        if cfg.save_last_every_epoch:
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "grpo_last",
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "grpo_last.pt"),
            )

        cuda_cleanup()

    model.load_state_dict(best_state, strict=True)
    return {"best_acc": best_acc, "best_state": best_state}


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = HLCMHardNegGRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Using normalizer:", cfg.use_normalizer)

    train_hf, eval_hf = load_mawps_local(cfg)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(eval_hf))))
    answer_pool_meta = build_answer_pool_metadata(answer_pool)

    print(f"[pool] unique answers in hard-negative pool: {len(answer_pool_meta)}")

    train_rows = build_or_load_cached_split(cfg, "train", train_hf, conceptizer, answer_pool_meta)
    eval_rows = build_or_load_cached_split(cfg, "validation", eval_hf, conceptizer, answer_pool_meta)

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )
    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)

    total_params = sum(p.numel() for p in hlcm.parameters())
    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Train rows: {len(train_ds)}")
    print(f"Eval rows: {len(eval_ds)}")

    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    print("\n========== STAGE 1: SFT ==========")
    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=cfg.out_dir,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "sft_best_acc": sft_result["best_acc"],
            "config": asdict(cfg),
        },
        os.path.join(cfg.out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()

    grpo_result = None
    if cfg.run_grpo:
        ref_logits_path = ref_logits_file_path(cfg, "train")
        ref_logits_rows = precompute_reference_logits(
            model=hlcm,
            dataset=train_ds,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
            out_path=ref_logits_path,
        )
        cuda_cleanup()

        print("\n========== STAGE 2: GRPO ==========")
        grpo_result = run_stage_grpo_hlcm_cached_ref(
            model=hlcm,
            train_loader=train_loader,
            eval_loader=eval_loader,
            ref_logits_rows=ref_logits_rows,
            out_dir=cfg.out_dir,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            device=device,
        )

        hlcm.load_state_dict(grpo_result["best_state"], strict=True)
        del grpo_result["best_state"]
        cuda_cleanup()

    final_eval = evaluate_hlcm_detailed(hlcm, eval_loader, mu, sigma, cfg)
    dump_eval_predictions(
        model=hlcm,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        out_path=os.path.join(cfg.out_dir, "final_eval_predictions.json"),
    )

    summary = {
        "sft_best_acc": None if 'sft_result' not in locals() else sft_result["best_acc"],
        "grpo_best_acc": None if grpo_result is None else grpo_result["best_acc"],
        "final_val_loss": final_eval["loss"],
        "final_val_acc": final_eval["acc"],
        "final_val_chance_acc": final_eval["chance_acc"],
        "config": asdict(cfg),
    }
    safe_json_dump(summary, os.path.join(cfg.out_dir, "final_summary.json"))

    print("\nDone.")
    print(f"SFT best acc:  {summary['sft_best_acc']:.4f}")
    if grpo_result is not None:
        print(f"GRPO best acc: {summary['grpo_best_acc']:.4f}")
    print(f"Final val acc: {summary['final_val_acc']:.4f}")
    print("Outputs in:", cfg.out_dir)


if __name__ == "__main__":
    main()

Device: cuda:0
Using normalizer: False


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pool] unique answers in hard-negative pool: 359
[cache] building train


cache:train: 100%|██████████████████████████| 1417/1417 [05:09<00:00,  4.59it/s]


...

[GRPO][epoch 2/4] train_loss=-0.0652 train_reward=0.1782 val_loss=1.8206 val_acc=0.3521 chance=0.1250


grpo 3/4: 100%|█| 355/355 [05:43<00:00,  1.03it/s, kl=0.1930, loss=-0.0646, lr=1


[GRPO][epoch 3/4] train_loss=-0.0646 train_reward=0.1699 val_loss=1.8459 val_acc=0.3465 chance=0.1250


grpo 4/4: 100%|█| 355/355 [05:10<00:00,  1.14it/s, kl=0.0191, loss=-0.0834, lr=0


[GRPO][epoch 4/4] train_loss=-0.0834 train_reward=0.1770 val_loss=1.8286 val_acc=0.3521 chance=0.1250

Done.
SFT best acc:  0.3437
GRPO best acc: 0.3577
Final val acc: 0.3577
Outputs in: runs/hlcm_mawps_hardneg_grpo_v1


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import uuid
import random
import hashlib
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


@dataclass
class EvalConfig:
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_mawps_hardneg_grpo_v1"
    cache_dir: str = "mawps_cached_hardneg_v1"

    base_ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    eval_ckpt_path: str = "runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt"

    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    instruction: str = (
        "Solve the math word problem.\n"
        "Return only the final numeric answer."
    )

    eval_batch_size: int = 8
    num_workers: int = 0
    mcq_logit_temperature: float = 0.1

    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def load_normalizer(path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        return None, None
    if not path or not os.path.exists(path):
        return None, None
    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        pass
    if re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s):
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None
    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""
    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")
    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        return canonicalize_numeric_str(text.split("####")[-1].strip())

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    candidates = frac_matches + dec_matches

    if candidates:
        best = None
        best_pos = -1
        for m in candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos
        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def numeric_equal(a: str, b: str, tol: float = 1e-6) -> bool:
    fa = safe_float_from_fraction_or_decimal(a)
    fb = safe_float_from_fraction_or_decimal(b)
    if fa is not None and fb is not None:
        return abs(fa - fb) <= tol
    return canonicalize_numeric_str(a) == canonicalize_numeric_str(b)


def try_parse_float(s: str) -> Optional[float]:
    try:
        x = float(str(s).strip().replace(",", ""))
        return x if math.isfinite(x) else None
    except Exception:
        return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return x is not None and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    return 0 if "." not in s else len(s.split(".")[-1])


def normalize_local_mawps_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = extract_final_numeric_answer(answer)
    return row


def load_mawps_local(cfg: EvalConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )
    return raw["train"], raw["validation"]


def build_mawps_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts):
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        out = (hs * attn).sum(dim=1) / attn.sum(dim=1).clamp_min(1.0)
        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str):
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []
        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str):
        chunks = self._pack_into_chunk_texts(text)
        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunks:
            return seq, pad

        vecs = []
        for i in range(0, len(chunks), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunks[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_mawps_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small.")
    return out


def build_answer_pool_metadata(answer_pool):
    meta = []
    for ans in answer_pool:
        x = try_parse_float(ans)
        meta.append({
            "answer": ans,
            "value": x,
            "abs_value": abs(x) if x is not None else float("inf"),
            "is_integerish": is_integerish_str(ans),
            "decimals": num_decimal_places(ans),
            "str_len": len(ans),
        })
    return meta


def deterministic_fallback_order_key(candidate_answer, gold_answer, question):
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return stable_int_hash(seed_text), candidate_answer


def choose_hard_negative_answers(gold_answer, question, answer_pool_meta, k_neg, close_k=64):
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []
    for item in answer_pool_meta:
        cand = item["answer"]
        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append({
            "answer": cand,
            "dist": dist,
            "same_sign": 1 if same_sign(gold_value, val) else 0,
            "same_int": 1 if item["is_integerish"] == gold_is_int else 0,
            "same_dec": 1 if item["decimals"] == gold_decimals else 0,
            "len_gap": abs(item["str_len"] - gold_len),
            "abs_gap": abs_gap,
            "fallback_hash": fallback_hash,
        })

    candidates.sort(key=lambda z: (
        z["dist"],
        -z["same_sign"],
        -z["same_int"],
        -z["same_dec"],
        z["abs_gap"],
        z["len_gap"],
        z["fallback_hash"],
        z["answer"],
    ))

    selected, seen = [], set()

    for item in candidates[:max(k_neg * 4, close_k)] + candidates:
        ans = item["answer"]
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)
        if len(selected) >= k_neg:
            return selected[:k_neg]

    return selected[:k_neg]


def cache_file_path(cfg: EvalConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def normalize_mawps_example_for_ranking(ex, cfg, answer_pool_meta):
    ex = normalize_local_mawps_row(ex)
    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_mawps_prompt(question, cfg.instruction)
    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(cfg, split_name, hf_split, conceptizer, answer_pool_meta):
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {split_name}")
    rows, skipped = [], 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_mawps_example_for_ranking(
                ex, cfg, answer_pool_meta
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)
            c_seqs, c_pads = [], []

            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(len(c_seqs), dtype=torch.bool),
                "label": int(label),
                "num_choices": len(c_seqs),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
                "question_text": q_text,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch):
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(x["choices"].size(0) for x in batch)

    q = torch.stack([x["q"] for x in batch], dim=0)
    qmask = torch.stack([x["qmask"] for x in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts, gold_answers, question_texts = [], [], []

    for i, x in enumerate(batch):
        K = x["choices"].size(0)
        choices[i, :K] = x["choices"]
        cmask[i, :K] = x["cmask"]
        choice_mask[i, :K] = x["choice_mask"]
        labels[i] = x["label"]
        idxs[i] = x["idx"]
        choice_texts.append(x["choice_texts"])
        gold_answers.append(x["gold_answer"])
        question_texts.append(x["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


def build_hlcm_from_cfg(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_best_model(cfg, device):
    model = build_hlcm_from_cfg(cfg).to(device)

    if os.path.exists(cfg.base_ckpt_path):
        base_obj = torch.load(cfg.base_ckpt_path, map_location="cpu")
        base_state = base_obj["model"] if isinstance(base_obj, dict) and "model" in base_obj else base_obj
        model.load_state_dict(base_state, strict=False)

    if not os.path.exists(cfg.eval_ckpt_path):
        raise FileNotFoundError(f"grpo_best.pt not found: {cfg.eval_ckpt_path}")

    ckpt = torch.load(cfg.eval_ckpt_path, map_location="cpu")
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded eval checkpoint: {cfg.eval_ckpt_path}")
    if isinstance(ckpt, dict):
        print(f"[load] stage={ckpt.get('stage')}")
        print(f"[load] best_val_acc={ckpt.get('best_val_acc')}")
    print(f"[load] missing={len(missing)} unexpected={len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, ckpt if isinstance(ckpt, dict) else {}


def sanitize_tensor(x, clamp_value, replace_nonfinite_with_zero):
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    return torch.clamp(x, -clamp_value, clamp_value)


def hlcm_encode_full_memory_safe(model, x):
    if not hasattr(model, "encode_inputs") or not hasattr(model, "layers"):
        return model(x)

    h = model.encode_inputs(x)
    for layer in list(model.layers):
        h = layer(h)
    return h


def hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg):
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h = hlcm_encode_full_memory_safe(model, x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    return h_tan


def hlcm_last_tangent(model, x, pad_mask, mu, sigma, cfg):
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)
    pad_mask = pad_mask.to(h_tan.device)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(model, batch, mu, sigma, cfg):
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu, sigma, cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        flat = choices[:, k0:k1].reshape(B * (k1 - k0), T, D)
        flat_mask = cmask[:, k0:k1].reshape(B * (k1 - k0), T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu, sigma, cfg)
        e_c = e_c.reshape(B, k1 - k0, -1)
        e_c = F.normalize(e_c, dim=-1)

        logits_list.append(torch.einsum("bd,bkd->bk", e_q, e_c))

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)
    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)
    return logits


@torch.no_grad()
def evaluate_precision_recall(model, loader, mu, sigma, cfg):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0

    hit_at_1 = 0
    hit_at_2 = 0
    hit_at_3 = 0
    hit_at_5 = 0
    mrr_sum = 0.0

    pred_rows = []

    for batch in tqdm(loader, desc="eval grpo_best"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=1)
        ranked = torch.argsort(logits, dim=1, descending=True)

        bs = labels.size(0)
        total += bs
        total_loss += float(loss.item()) * bs
        correct += int((preds == labels).sum().item())

        probs = F.softmax(logits, dim=-1)

        for i in range(bs):
            valid_k = int(batch["choice_mask"][i].sum().item())
            gold_idx = int(labels[i].item())
            pred_idx = int(preds[i].item())
            rank_list = ranked[i, :valid_k].detach().cpu().tolist()
            rank = rank_list.index(gold_idx) + 1

            hit_at_1 += int(rank <= 1)
            hit_at_2 += int(rank <= 2)
            hit_at_3 += int(rank <= 3)
            hit_at_5 += int(rank <= min(5, valid_k))
            mrr_sum += 1.0 / rank

            choices = batch["choice_texts"][i]
            pred_answer = choices[pred_idx]
            gold_answer = batch["gold_answers"][i]

            pred_rows.append({
                "question_text": batch["question_texts"][i],
                "gold_answer": gold_answer,
                "pred_answer": pred_answer,
                "gold_choice_index": gold_idx,
                "pred_choice_index": pred_idx,
                "rank_of_gold": rank,
                "correct_top1": int(pred_idx == gold_idx),
                "numeric_equal_pred_gold": int(numeric_equal(pred_answer, gold_answer)),
                "choice_texts": choices[:valid_k],
                "choice_logits": [float(x) for x in logits[i, :valid_k].detach().cpu().tolist()],
                "choice_probs": [float(x) for x in probs[i, :valid_k].detach().cpu().tolist()],
                "ranked_indices": rank_list,
            })

    metrics = {
        "num_examples": total,
        "loss": total_loss / max(1, total),
        "accuracy_top1": correct / max(1, total),

        "precision@1": hit_at_1 / max(1, total),
        "recall@1": hit_at_1 / max(1, total),

        "precision@2": (hit_at_2 / 2.0) / max(1, total),
        "recall@2": hit_at_2 / max(1, total),

        "precision@3": (hit_at_3 / 3.0) / max(1, total),
        "recall@3": hit_at_3 / max(1, total),

        "precision@5": (hit_at_5 / 5.0) / max(1, total),
        "recall@5": hit_at_5 / max(1, total),

        "mrr": mrr_sum / max(1, total),
        "chance_accuracy": 1.0 / max(1, cfg.num_choices),
    }

    return metrics, pred_rows


def main():
    cfg = EvalConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("EVAL ONLY: no training will run.")
    print("Using checkpoint:", cfg.eval_ckpt_path)
    print("Using normalizer:", cfg.use_normalizer)

    train_hf, eval_hf = load_mawps_local(cfg)

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(eval_hf))))

    answer_pool_meta = build_answer_pool_metadata(answer_pool)
    print(f"[pool] unique answers: {len(answer_pool_meta)}")

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
        answer_pool_meta=answer_pool_meta,
    )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_info = load_grpo_best_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    metrics, predictions = evaluate_precision_recall(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
    )

    result = {
        "checkpoint": cfg.eval_ckpt_path,
        "checkpoint_stage": ckpt_info.get("stage", None),
        "checkpoint_best_val_acc": ckpt_info.get("best_val_acc", None),
        "metrics": metrics,
        "note": "For this ranking setup there is exactly one correct answer per problem. Therefore recall@1 equals top-1 accuracy.",
    }

    metrics_path = os.path.join(cfg.out_dir, "grpo_best_precision_recall_eval.json")
    predictions_path = os.path.join(cfg.out_dir, "grpo_best_precision_recall_predictions.json")

    safe_json_dump(result, metrics_path)
    safe_json_dump(predictions, predictions_path)

    print(json.dumps(result, indent=2))
    print("Saved metrics to:", metrics_path)
    print("Saved predictions to:", predictions_path)


if __name__ == "__main__":
    main()

Device: cuda:0
EVAL ONLY: no training will run.
Using checkpoint: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
Using normalizer: False
[pool] unique answers: 359


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt
[load] loaded eval checkpoint: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
[load] stage=grpo_best
[load] best_val_acc=0.35774647887323946
[load] missing=0 unexpected=0


eval grpo_best: 100%|███████████████████████████████████████████████| 45/45 [00:24<00:00,  1.86it/s]


{
  "checkpoint": "runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt",
  "checkpoint_stage": "grpo_best",
  "checkpoint_best_val_acc": 0.35774647887323946,
  "metrics": {
    "num_examples": 355,
    "loss": 1.833815501777219,
    "accuracy_top1": 0.35774647887323946,
    "precision@1": 0.35774647887323946,
    "recall@1": 0.35774647887323946,
    "precision@2": 0.2704225352112676,
    "recall@2": 0.5408450704225352,
    "precision@3": 0.22723004694835683,
    "recall@3": 0.6816901408450704,
    "precision@5": 0.1673239436619718,
    "recall@5": 0.8366197183098592,
    "mrr": 0.5554325955734405,
    "chance_accuracy": 0.125
  },
  "note": "For this ranking setup there is exactly one correct answer per problem. Therefore recall@1 equals top-1 accuracy."
}
Saved metrics to: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_precision_recall_eval.json
Saved predictions to: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_precision_recall_predictions.json


In [1]:
# eval_grpo_best_mawps_metrics.py
# No training. Loads grpo_best.pt and evaluates precision, recall, F1, Brier, and ECE.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


@dataclass
class EvalConfig:
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_mawps_hardneg_grpo_v1"
    cache_dir: str = "mawps_cached_hardneg_v1"

    checkpoint_path: str = "runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt"
    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    instruction: str = (
        "Solve the math word problem.\n"
        "Return only the final numeric answer."
    )

    eval_batch_size: int = 8
    num_workers: int = 0
    use_bf16: bool = True
    mcq_logit_temperature: float = 0.1

    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    ece_bins: int = 15
    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def load_normalizer(normalizer_path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        return None, None
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def assert_finite(name: str, x: torch.Tensor):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(f"{name} has non-finite values; bad_count={bad}; shape={tuple(x.shape)}")


def sanitize_tensor(x: torch.Tensor, clamp_value: float, replace_nonfinite_with_zero: bool) -> torch.Tensor:
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    return torch.clamp(x, -clamp_value, clamp_value)


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        pass

    if re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s):
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None
    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if not s:
        return ""
    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")
    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        return canonicalize_numeric_str(text.split("####")[-1].strip())

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    all_candidates = frac_matches + dec_matches

    if all_candidates:
        best = None
        best_pos = -1
        for m in all_candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos
        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def try_parse_float(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    try:
        x = float(s)
        if math.isfinite(x):
            return float(x)
    except Exception:
        return None
    return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return (x is not None) and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    if "." not in s:
        return 0
    return len(s.split(".")[-1])


def normalize_local_mawps_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    final_numeric_answer = extract_final_numeric_answer(answer)

    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = final_numeric_answer
    return row


def load_mawps_local(cfg: EvalConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )
    return raw["train"], raw["validation"]


def build_mawps_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        summed = (hs * attn).sum(dim=1)
        denom = attn.sum(dim=1).clamp_min(1.0)
        out = summed / denom
        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_mawps_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small.")
    return out


def build_answer_pool_metadata(answer_pool: List[str]) -> List[Dict[str, Any]]:
    meta = []
    for ans in answer_pool:
        x = try_parse_float(ans)
        meta.append({
            "answer": ans,
            "value": x,
            "abs_value": abs(x) if x is not None else float("inf"),
            "is_integerish": is_integerish_str(ans),
            "decimals": num_decimal_places(ans),
            "str_len": len(ans),
        })
    return meta


def deterministic_fallback_order_key(candidate_answer: str, gold_answer: str, question: str) -> Tuple[int, str]:
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return (stable_int_hash(seed_text), candidate_answer)


def choose_hard_negative_answers(
    gold_answer: str,
    question: str,
    answer_pool_meta: List[Dict[str, Any]],
    k_neg: int,
    close_k: int = 64,
) -> List[str]:
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []
    for item in answer_pool_meta:
        cand = item["answer"]
        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        same_sign_flag = 1 if same_sign(gold_value, val) else 0
        same_int_flag = 1 if item["is_integerish"] == gold_is_int else 0
        same_dec_flag = 1 if item["decimals"] == gold_decimals else 0
        len_gap = abs(item["str_len"] - gold_len)
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append({
            "answer": cand,
            "dist": dist,
            "same_sign": same_sign_flag,
            "same_int": same_int_flag,
            "same_dec": same_dec_flag,
            "len_gap": len_gap,
            "abs_gap": abs_gap,
            "fallback_hash": fallback_hash,
        })

    candidates.sort(
        key=lambda z: (
            z["dist"],
            -z["same_sign"],
            -z["same_int"],
            -z["same_dec"],
            z["abs_gap"],
            z["len_gap"],
            z["fallback_hash"],
            z["answer"],
        )
    )

    selected = []
    seen = set()
    for item in candidates[:max(k_neg * 4, close_k)] + candidates:
        ans = item["answer"]
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)
        if len(selected) >= k_neg:
            break

    return selected[:k_neg]


def cache_file_path(cfg: EvalConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def normalize_mawps_example_for_ranking(
    ex: Dict[str, Any],
    cfg: EvalConfig,
    answer_pool_meta: List[Dict[str, Any]],
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_mawps_row(ex)
    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_mawps_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(
    cfg: EvalConfig,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
    answer_pool_meta: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    print(f"[cache] building {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_mawps_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool_meta=answer_pool_meta,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)
            c_seqs, c_pads = [], []

            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
                "question_text": q_text,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []
    question_texts = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]
        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])
        question_texts.append(item["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_best_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    if not os.path.exists(cfg.checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.checkpoint_path}")

    model = build_hlcm_from_cfg(cfg).to(device)
    obj = torch.load(cfg.checkpoint_path, map_location="cpu")

    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded eval checkpoint: {cfg.checkpoint_path}")
    print(f"[load] stage: {obj.get('stage', 'unknown') if isinstance(obj, dict) else 'unknown'}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    return model


def hlcm_encode_full_memory_safe(model: HyperbolicLCM, x: torch.Tensor) -> torch.Tensor:
    if not hasattr(model, "encode_inputs") or not hasattr(model, "layers"):
        return model(x)

    layers = list(model.layers)
    if len(layers) == 0:
        return model(x)

    h = model.encode_inputs(x)
    for layer in layers:
        h = layer(h)
    return h


def hlcm_tangent_sequence(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)

    if cfg.strict_finite_checks:
        assert_finite("input_x_before_norm", x)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h = hlcm_encode_full_memory_safe(model, x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    if cfg.strict_finite_checks:
        assert_finite("hlcm_h_tan", h_tan)

    return h_tan


def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)
    B, T, D = h_tan.shape
    pad_mask = pad_mask.to(h_tan.device)
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        ec = ec.reshape(B, (k1 - k0), -1)
        ec = F.normalize(ec, dim=-1)

        chunk_logits = torch.einsum("bd,bkd->bk", e_q, ec)
        logits_list.append(chunk_logits)

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def compute_precision_recall_f1_from_counts(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1


def compute_multiclass_prf(
    y_true: List[int],
    y_pred: List[int],
    num_classes: int,
) -> Dict[str, Any]:
    per_class = []
    macro_p = macro_r = macro_f1 = 0.0
    weighted_p = weighted_r = weighted_f1 = 0.0

    total_support = len(y_true)

    for c in range(num_classes):
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp == c)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != c and yp == c)
        fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp != c)
        support = sum(1 for yt in y_true if yt == c)

        p, r, f1 = compute_precision_recall_f1_from_counts(tp, fp, fn)

        per_class.append({
            "class": c,
            "precision": p,
            "recall": r,
            "f1": f1,
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        })

        macro_p += p
        macro_r += r
        macro_f1 += f1

        weighted_p += p * support
        weighted_r += r * support
        weighted_f1 += f1 * support

    macro_p /= max(num_classes, 1)
    macro_r /= max(num_classes, 1)
    macro_f1 /= max(num_classes, 1)

    weighted_p /= max(total_support, 1)
    weighted_r /= max(total_support, 1)
    weighted_f1 /= max(total_support, 1)

    correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
    accuracy = correct / max(total_support, 1)

    return {
        "accuracy": accuracy,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
        "per_class": per_class,
    }


def compute_ece(confidences: List[float], correctness: List[int], n_bins: int = 15) -> Dict[str, Any]:
    n = len(confidences)
    ece = 0.0
    mce = 0.0
    bins = []

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins

        if b == n_bins - 1:
            idx = [i for i, c in enumerate(confidences) if lo <= c <= hi]
        else:
            idx = [i for i, c in enumerate(confidences) if lo <= c < hi]

        count = len(idx)
        if count == 0:
            bins.append({
                "bin": b,
                "lower": lo,
                "upper": hi,
                "count": 0,
                "accuracy": None,
                "confidence": None,
                "gap": None,
            })
            continue

        acc = sum(correctness[i] for i in idx) / count
        conf = sum(confidences[i] for i in idx) / count
        gap = abs(acc - conf)

        ece += (count / max(n, 1)) * gap
        mce = max(mce, gap)

        bins.append({
            "bin": b,
            "lower": lo,
            "upper": hi,
            "count": count,
            "accuracy": acc,
            "confidence": conf,
            "gap": gap,
        })

    return {
        "ece": ece,
        "mce": mce,
        "bins": bins,
    }


@torch.no_grad()
def evaluate_with_precision_recall_brier_ece(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    model.eval()

    y_true = []
    y_pred = []
    confidences = []
    correctness = []
    prediction_rows = []

    total_loss = 0.0
    total_brier = 0.0
    total_nll = 0.0
    total_examples = 0
    total_chance = 0.0

    max_num_classes_seen = 0

    for batch in tqdm(loader, desc="evaluating"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        conf = probs.max(dim=-1).values

        B, K = logits.shape
        max_num_classes_seen = max(max_num_classes_seen, K)

        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        valid_float = choice_mask.float()
        brier_per_example = (((probs - one_hot) ** 2) * valid_float).sum(dim=1)

        nll_per_example = -torch.log(probs.gather(1, labels.view(-1, 1)).squeeze(1).clamp_min(1e-12))

        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_brier += float(brier_per_example.sum().item())
        total_nll += float(nll_per_example.sum().item())
        total_examples += bs

        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()
        for k in choice_counts:
            total_chance += 1.0 / max(1, int(k))

        labels_cpu = labels.detach().cpu().tolist()
        preds_cpu = preds.detach().cpu().tolist()
        conf_cpu = conf.detach().cpu().tolist()
        correct_cpu = (preds == labels).detach().cpu().int().tolist()

        y_true.extend(labels_cpu)
        y_pred.extend(preds_cpu)
        confidences.extend([float(x) for x in conf_cpu])
        correctness.extend([int(x) for x in correct_cpu])

        probs_cpu = probs.detach().cpu()
        logits_cpu = logits.detach().cpu()

        for i in range(bs):
            valid_k = int(choice_counts[i])
            p = int(preds_cpu[i])
            g = int(labels_cpu[i])
            choices = batch["choice_texts"][i]

            prediction_rows.append({
                "idx": int(batch["idx"][i].item()),
                "question_text": batch["question_texts"][i],
                "gold_answer": batch["gold_answers"][i],
                "pred_answer": choices[p],
                "gold_choice_index": g,
                "pred_choice_index": p,
                "correct": int(p == g),
                "confidence": float(conf_cpu[i]),
                "brier": float(brier_per_example[i].detach().cpu().item()),
                "nll": float(nll_per_example[i].detach().cpu().item()),
                "choice_texts": choices,
                "choice_logits": [float(x) for x in logits_cpu[i, :valid_k].tolist()],
                "choice_probs": [float(x) for x in probs_cpu[i, :valid_k].tolist()],
            })

    prf = compute_multiclass_prf(y_true, y_pred, num_classes=max_num_classes_seen)
    ece_obj = compute_ece(confidences, correctness, n_bins=cfg.ece_bins)

    metrics = {
        "num_examples": total_examples,
        "loss": total_loss / max(total_examples, 1),
        "nll": total_nll / max(total_examples, 1),
        "accuracy": prf["accuracy"],
        "chance_accuracy": total_chance / max(total_examples, 1),

        "brier_score": total_brier / max(total_examples, 1),

        "ece": ece_obj["ece"],
        "mce": ece_obj["mce"],
        "ece_bins": cfg.ece_bins,
        "ece_bin_details": ece_obj["bins"],

        "macro_precision": prf["macro_precision"],
        "macro_recall": prf["macro_recall"],
        "macro_f1": prf["macro_f1"],
        "weighted_precision": prf["weighted_precision"],
        "weighted_recall": prf["weighted_recall"],
        "weighted_f1": prf["weighted_f1"],
        "per_class": prf["per_class"],

        "config": asdict(cfg),
    }

    return metrics, prediction_rows


def main():
    cfg = EvalConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Checkpoint:", cfg.checkpoint_path)
    print("Using normalizer:", cfg.use_normalizer)

    train_hf, eval_hf = load_mawps_local(cfg)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(eval_hf))))

    answer_pool_meta = build_answer_pool_metadata(answer_pool)
    print(f"[pool] unique answers: {len(answer_pool_meta)}")

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
        answer_pool_meta=answer_pool_meta,
    )

    del conceptizer
    cuda_cleanup()

    eval_ds = CachedMCQDataset(eval_rows)
    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model = load_grpo_best_hlcm(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    metrics, prediction_rows = evaluate_with_precision_recall_brier_ece(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
    )

    metrics_path = os.path.join(cfg.out_dir, "grpo_best_eval_metrics_precision_recall_brier_ece.json")
    preds_path = os.path.join(cfg.out_dir, "grpo_best_eval_predictions.json")
    csv_path = os.path.join(cfg.out_dir, "grpo_best_eval_metrics_precision_recall_brier_ece.csv")

    safe_json_dump(metrics, metrics_path)
    safe_json_dump(prediction_rows, preds_path)

    flat_metrics = {
        "num_examples": metrics["num_examples"],
        "loss": metrics["loss"],
        "nll": metrics["nll"],
        "accuracy": metrics["accuracy"],
        "chance_accuracy": metrics["chance_accuracy"],
        "brier_score": metrics["brier_score"],
        "ece": metrics["ece"],
        "mce": metrics["mce"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "macro_f1": metrics["macro_f1"],
        "weighted_precision": metrics["weighted_precision"],
        "weighted_recall": metrics["weighted_recall"],
        "weighted_f1": metrics["weighted_f1"],
    }
    write_single_row_csv(csv_path, flat_metrics)

    print("\n========== GRPO BEST EVAL ==========")
    print(f"Examples:           {metrics['num_examples']}")
    print(f"Loss:               {metrics['loss']:.6f}")
    print(f"NLL:                {metrics['nll']:.6f}")
    print(f"Accuracy:           {metrics['accuracy']:.6f}")
    print(f"Chance accuracy:    {metrics['chance_accuracy']:.6f}")
    print(f"Precision macro:    {metrics['macro_precision']:.6f}")
    print(f"Recall macro:       {metrics['macro_recall']:.6f}")
    print(f"F1 macro:           {metrics['macro_f1']:.6f}")
    print(f"Precision weighted: {metrics['weighted_precision']:.6f}")
    print(f"Recall weighted:    {metrics['weighted_recall']:.6f}")
    print(f"F1 weighted:        {metrics['weighted_f1']:.6f}")
    print(f"Brier score:        {metrics['brier_score']:.6f}")
    print(f"ECE:                {metrics['ece']:.6f}")
    print(f"MCE:                {metrics['mce']:.6f}")
    print("\nSaved:")
    print(metrics_path)
    print(preds_path)
    print(csv_path)


if __name__ == "__main__":
    main()

Device: cuda:0
Checkpoint: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
Using normalizer: False


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pool] unique answers: 359
[cache] loading mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt
[load] loaded eval checkpoint: runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
[load] stage: grpo_best
[load] missing keys: 0
[load] unexpected keys: 0


evaluating: 100%|███████████████████████████████████████████████████| 45/45 [00:20<00:00,  2.23it/s]



========== GRPO BEST EVAL ==========
Examples:           355
Loss:               1.833816
NLL:                1.833815
Accuracy:           0.357746
Chance accuracy:    0.125000
Precision macro:    0.125000
Recall macro:       0.044718
F1 macro:           0.065871
Precision weighted: 1.000000
Recall weighted:    0.357746
F1 weighted:        0.526971
Brier score:        0.801387
ECE:                0.184779
MCE:                0.637565

Saved:
runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_eval_metrics_precision_recall_brier_ece.json
runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_eval_predictions.json
runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_eval_metrics_precision_recall_brier_ece.csv


In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import json
import math
import time
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_mawps_hardneg_grpo_v1"
    cache_dir: str = "mawps_cached_hardneg_v1"

    grpo_ckpt_path: str = "runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt"

    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    instruction: str = (
        "Solve the math word problem.\n"
        "Return only the final numeric answer."
    )

    eval_batch_size: int = 8
    num_workers: int = 0

    mcq_logit_temperature: float = 0.1

    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    seed: int = 42
    prefer_gpu_index: int = 0

    split: str = "validation"   # "validation" or "train"
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        print("[normalizer] disabled")
        return None, None

    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def sanitize_tensor(x: torch.Tensor, clamp_value: float, replace_nonfinite_with_zero: bool) -> torch.Tensor:
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    return torch.clamp(x, -clamp_value, clamp_value)


def assert_finite(name: str, x: torch.Tensor):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(f"{name} has non-finite values; bad_count={bad}; shape={tuple(x.shape)}")


# ============================================================
# NUMERIC UTILS
# ============================================================

def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None

    try:
        return float(s)
    except Exception:
        pass

    if re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s):
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None

    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""

    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")

    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""

    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        candidate = text.split("####")[-1].strip()
        return canonicalize_numeric_str(candidate)

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    all_candidates = frac_matches + dec_matches

    if all_candidates:
        best = None
        best_pos = -1

        for m in all_candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos

        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def try_parse_float(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    try:
        x = float(s)
        if math.isfinite(x):
            return float(x)
    except Exception:
        return None
    return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return (x is not None) and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    if "." not in s:
        return 0
    return len(s.split(".")[-1])


# ============================================================
# DATA
# ============================================================

def normalize_local_mawps_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)

    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    final_numeric_answer = extract_final_numeric_answer(answer)

    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = final_numeric_answer

    return row


def load_mawps_local(cfg: InferenceConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    return raw["train"], raw["validation"]


def build_mawps_prompt(question: str, instruction: str) -> str:
    return "\n".join(
        [
            instruction.strip(),
            "",
            "Problem:",
            question.strip(),
            "",
            "Final Answer:",
        ]
    )


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        summed = (hs * attn).sum(dim=1)
        denom = attn.sum(dim=1).clamp_min(1.0)
        out = summed / denom

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break

        return chunks[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# HARD NEGATIVE ANSWER POOL
# ============================================================

def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()

    for ex in hf_split:
        ex = normalize_local_mawps_row(ex)
        ans = ex["final_numeric_answer"]

        if ans:
            pool.add(ans)

    out = sorted(pool)

    if len(out) < 2:
        raise ValueError("Answer pool is too small to build negative candidates.")

    return out


def build_answer_pool_metadata(answer_pool: List[str]) -> List[Dict[str, Any]]:
    meta = []

    for ans in answer_pool:
        x = try_parse_float(ans)

        meta.append(
            {
                "answer": ans,
                "value": x,
                "abs_value": abs(x) if x is not None else float("inf"),
                "is_integerish": is_integerish_str(ans),
                "decimals": num_decimal_places(ans),
                "str_len": len(ans),
            }
        )

    return meta


def deterministic_fallback_order_key(candidate_answer: str, gold_answer: str, question: str) -> Tuple[int, str]:
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return stable_int_hash(seed_text), candidate_answer


def choose_hard_negative_answers(
    gold_answer: str,
    question: str,
    answer_pool_meta: List[Dict[str, Any]],
    k_neg: int,
    close_k: int = 64,
) -> List[str]:
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []

    for item in answer_pool_meta:
        cand = item["answer"]

        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        same_sign_flag = 1 if same_sign(gold_value, val) else 0
        same_int_flag = 1 if item["is_integerish"] == gold_is_int else 0
        same_dec_flag = 1 if item["decimals"] == gold_decimals else 0
        len_gap = abs(item["str_len"] - gold_len)
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append(
            {
                "answer": cand,
                "dist": dist,
                "same_sign": same_sign_flag,
                "same_int": same_int_flag,
                "same_dec": same_dec_flag,
                "len_gap": len_gap,
                "abs_gap": abs_gap,
                "fallback_hash": fallback_hash,
            }
        )

    candidates.sort(
        key=lambda z: (
            z["dist"],
            -z["same_sign"],
            -z["same_int"],
            -z["same_dec"],
            z["abs_gap"],
            z["len_gap"],
            z["fallback_hash"],
            z["answer"],
        )
    )

    close_candidates = candidates[:max(k_neg * 4, close_k)]

    if len(close_candidates) == 0:
        raise ValueError("No negative answers available.")

    selected = []
    seen = set()

    def add_if_new(ans: str):
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)

    for item in close_candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    for item in candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    return selected[:k_neg]


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: InferenceConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"

    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt",
    )


def normalize_mawps_example_for_ranking(
    ex: Dict[str, Any],
    cfg: InferenceConfig,
    answer_pool_meta: List[Dict[str, Any]],
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_mawps_row(ex)

    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")

    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_mawps_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0

    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
    answer_pool_meta: List[Dict[str, Any]],
):
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(path)

    if conceptizer is None:
        raise RuntimeError("Conceptizer is required because cache is missing.")

    print(f"[cache] building {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_mawps_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool_meta=answer_pool_meta,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(choices.size(0), dtype=torch.bool)

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": choices,
                    "cmask": cmask,
                    "choice_mask": choice_mask,
                    "label": int(label),
                    "num_choices": int(choices.size(0)),
                    "gold_answer": gold_answer,
                    "choice_texts": choice_texts,
                    "question_text": q_text,
                }
            )

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []
    question_texts = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])
        question_texts.append(item["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_best_model(cfg: InferenceConfig, device: torch.device):
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(cfg.grpo_ckpt_path)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_best.pt does not contain key 'model'.")

    model = build_hlcm_from_cfg(cfg).to(device)
    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded model from {cfg.grpo_ckpt_path}")
    print(f"[load] stage: {obj.get('stage', 'unknown')}")
    print(f"[load] best_val_acc: {obj.get('best_val_acc', 'unknown')}")
    print(f"[load] epoch: {obj.get('epoch', 'unknown')}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model, obj


# ============================================================
# HLCM FORWARD
# ============================================================

@torch.no_grad()
def hlcm_encode_full(model: HyperbolicLCM, x: torch.Tensor) -> torch.Tensor:
    return model(x)


@torch.no_grad()
def hlcm_tangent_sequence(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if cfg.strict_finite_checks:
        assert_finite("input_x_before_norm", x)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    h = model(x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    if cfg.strict_finite_checks:
        assert_finite("hlcm_h_tan", h_tan)

    return h_tan


@torch.no_grad()
def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    pad_mask = pad_mask.to(h_tan.device)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: InferenceConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        ec = ec.reshape(B, (k1 - k0), -1)
        ec = F.normalize(ec, dim=-1)

        chunk_logits = torch.einsum("bd,bkd->bk", e_q, ec)
        logits_list.append(chunk_logits)

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)

    return logits


# ============================================================
# INFERENCE + TIMING
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model,
    loader,
    mu,
    sigma,
    cfg,
    save_path: Optional[str] = None,
):
    model.eval()
    device = next(model.parameters()).device

    total = 0
    correct = 0
    total_loss = 0.0
    total_chance = 0.0

    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference MAWPS-{cfg.split}"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)

        labels = batch["label"].to(logits.device, non_blocking=True)
        loss = F.cross_entropy(logits, labels)

        preds = logits.argmax(dim=1)

        bs = labels.size(0)
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        total += int(bs)
        correct += int((preds == labels).sum().item())
        total_loss += float(loss.item()) * int(bs)

        for k in choice_counts:
            total_chance += 1.0 / max(1, int(k))

        probs = F.softmax(logits, dim=-1)

        for i in range(bs):
            valid_k = int(choice_counts[i])
            pred_i = int(preds[i].item())
            gold_i = int(labels[i].item())
            choices_i = batch["choice_texts"][i]

            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold_answer": batch["gold_answers"][i],
                    "pred_answer": choices_i[pred_i],
                    "gold_choice_index": gold_i,
                    "pred_choice_index": pred_i,
                    "correct": int(pred_i == gold_i),
                    "choice_texts": choices_i,
                    "choice_logits": [float(x) for x in logits[i, :valid_k].detach().cpu().tolist()],
                    "choice_probs": [float(x) for x in probs[i, :valid_k].detach().cpu().tolist()],
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "chance_accuracy": total_chance / max(total, 1),
        "chance_accuracy_percent": 100.0 * total_chance / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        safe_json_dump(results, save_path)
        print(f"[save] results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_mawps(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[checkpoint]", cfg.grpo_ckpt_path)
    print("[use_normalizer]", cfg.use_normalizer)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    train_hf, val_hf = load_mawps_local(cfg)

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(val_hf))))

    answer_pool_meta = build_answer_pool_metadata(answer_pool)
    print(f"[pool] unique answers in hard-negative pool: {len(answer_pool_meta)}")

    if cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    elif cfg.split in {"validation", "val", "test"}:
        hf_split = val_hf
        split_name = "validation"
    else:
        raise ValueError("cfg.split must be 'train' or 'validation'.")

    cache_path = cache_file_path(cfg, split_name)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=torch.device(cfg.conceptizer_device),
        )

        t_cache = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=conceptizer,
            answer_pool_meta=answer_pool_meta,
        )

        cache_build_time_sec = time.perf_counter() - t_cache

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=None,
            answer_pool_meta=answer_pool_meta,
        )

    print(f"[data] split={split_name}, examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_obj = load_grpo_best_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    save_path = os.path.join(cfg.out_dir, f"grpo_best_inference_{split_name}_results.json")

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "checkpoint_stage": ckpt_obj.get("stage", None),
        "checkpoint_best_val_acc": ckpt_obj.get("best_val_acc", None),
        "checkpoint_epoch": ckpt_obj.get("epoch", None),
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "chance_accuracy": results["chance_accuracy"],
        "chance_accuracy_percent": results["chance_accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
        "config": asdict(cfg),
    }

    summary_path = os.path.join(cfg.out_dir, f"grpo_best_inference_{split_name}_summary.json")
    safe_json_dump(summary, summary_path)

    print("\n==================== GRPO BEST INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [2]:
cfg = InferenceConfig(
    grpo_ckpt_path="runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt",

    hf_train_file="/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet",
    hf_validation_file="/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet",

    out_dir="runs/hlcm_mawps_hardneg_grpo_v1",
    cache_dir="mawps_cached_hardneg_v1",

    split="validation",
    eval_batch_size=8,

    use_normalizer=False,
    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_mawps(cfg)

[device] cuda:0
[checkpoint] runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
[use_normalizer] False
[pool] unique answers in hard-negative pool: 359
[cache] not found: mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building validation


cache:validation: 100%|███████████████████████████████████████████| 355/355 [01:21<00:00,  4.36it/s]


[cache] saved mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt (355 examples, skipped=0)
[data] split=validation, examples=355
[cache] mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt
[load] loaded model from runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt
[load] stage: grpo_best
[load] best_val_acc: 0.35774647887323946
[load] epoch: 1
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] disabled


Inference MAWPS-validation: 100%|███████████████████████████████████| 45/45 [00:19<00:00,  2.28it/s]


[save] results -> runs/hlcm_mawps_hardneg_grpo_v1/grpo_best_inference_validation_results.json

==================== GRPO BEST INFERENCE DONE ====================
{
  "split": "validation",
  "checkpoint": "runs/hlcm_mawps_hardneg_grpo_v1/grpo_best.pt",
  "checkpoint_stage": "grpo_best",
  "checkpoint_best_val_acc": 0.35774647887323946,
  "checkpoint_epoch": 1,
  "cache_file": "mawps_cached_hardneg_v1/validation_hardneg_tok256_seq8_K8.pt",
  "num_examples": 355,
  "correct": 127,
  "loss": 1.833815501777219,
  "accuracy": 0.35774647887323946,
  "accuracy_percent": 35.774647887323944,
  "chance_accuracy": 0.125,
  "chance_accuracy_percent": 12.5,
  "cache_build_time_sec": 81.53633231809363,
  "cache_build_time_hms": "00:01:21",
  "inference_time_sec": 19.833386173006147,
  "inference_time_hms": "00:00:19",
  "time_per_example_sec": 0.05586869344508774,
  "examples_per_second": 17.89911197731661,
  "config": {
    "hf_train_file": "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-

In [3]:
print("Accuracy:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

Accuracy: 35.774647887323944
Inference time: 19.833386173006147
Time/example: 0.05586869344508774
Examples/sec: 17.89911197731661


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class HLCMMAWPSConfig:
    datasets_to_run: Tuple[str, ...] = ("mawps_local",)

    # local parquet files
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_mawps_local_interpretable"
    cache_dir: str = "mawps_cached_features"
    ref_logits_dir: str = "mawps_cached_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch: MUST match pretrained checkpoint
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 1

    # loader / memory
    train_batch_size: int = 1
    eval_batch_size: int = 1
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT
    sft_epochs: int = 2
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO
    grpo_epochs: int = 1
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # policy / reward
    mcq_logit_temperature: float = 0.1
    grpo_policy_temperature: float = 1.0
    grpo_group_size: int = 2
    grpo_beta_kl: float = 0.02
    entropy_bonus: float = 0.001
    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # answer-ranking construction
    num_choices: int = 4
    choice_chunk_size: int = 1
    min_valid_choices: int = 2

    # prompt format
    instruction: str = (
        "Solve the math word problem.\n"
        "Return only the final numeric answer.\n"
        "Do not show steps.\n"
        "Do not explain."
    )

    # misc
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# NUMERIC NORMALIZATION (MAWPS)
# ============================================================

def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""

    try:
        x = float(s)
        if math.isfinite(x):
            if x.is_integer():
                return str(int(x))
            return str(round(x, 6)).rstrip("0").rstrip(".")
    except Exception:
        pass

    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""

    text = str(text).strip()

    if "####" in text:
        candidate = text.split("####")[-1].strip()
        return canonicalize_numeric_str(candidate)

    matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if matches:
        return canonicalize_numeric_str(matches[-1])

    return canonicalize_numeric_str(text)


def normalize_local_mawps_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    final_numeric_answer = extract_final_numeric_answer(answer)

    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = final_numeric_answer
    return row


def build_mawps_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")

    n_last = max(1, min(n_last, len(layers)))
    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# MAWPS LOCAL LOADING
# ============================================================

def load_mawps_local(cfg: HLCMMAWPSConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    train_split = raw["train"]
    eval_split = raw["validation"]
    test_split = None
    return train_split, eval_split, test_split


def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_mawps_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small to build negative candidates.")
    return out


def sample_negative_answers(
    gold_answer: str,
    answer_pool: List[str],
    k_neg: int,
    rng: random.Random,
) -> List[str]:
    candidates = [a for a in answer_pool if a != gold_answer]
    if len(candidates) == 0:
        raise ValueError("No negative answers available.")

    if len(candidates) >= k_neg:
        return rng.sample(candidates, k_neg)

    out = []
    while len(out) < k_neg:
        out.extend(rng.sample(candidates, min(len(candidates), k_neg - len(out))))
    return out[:k_neg]


def normalize_mawps_example_for_ranking(
    ex: Dict[str, Any],
    cfg: HLCMMAWPSConfig,
    answer_pool: List[str],
    rng: random.Random,
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_mawps_row(ex)

    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_mawps_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = sample_negative_answers(
        gold_answer=gold_answer,
        answer_pool=answer_pool,
        k_neg=k_neg,
        rng=rng,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(cfg: HLCMMAWPSConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def ref_logits_file_path(cfg: HLCMMAWPSConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def build_or_load_cached_split(
    cfg: HLCMMAWPSConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
    answer_pool: List[str],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if hf_split is None:
        return []

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0
    rng = random.Random(cfg.seed + (0 if split_name == "train" else 1000))

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_mawps_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool=answer_pool,
                rng=rng,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]
        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: HLCMMAWPSConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: HLCMMAWPSConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# MEMORY-SAFE HLCM FORWARD
# ============================================================

def get_trainable_layer_start(model: HyperbolicLCM) -> int:
    if not hasattr(model, "layers"):
        return 0

    layers = list(model.layers)
    if len(layers) == 0:
        return 0

    first_trainable = len(layers)
    for i, layer in enumerate(layers):
        has_grad = any(p.requires_grad for p in layer.parameters())
        if has_grad:
            first_trainable = i
            break
    return first_trainable


def hlcm_encode_full_memory_safe(
    model: HyperbolicLCM,
    x: torch.Tensor,
) -> torch.Tensor:
    if not hasattr(model, "encode_inputs") or not hasattr(model, "layers"):
        return model(x)

    layers = list(model.layers)
    if len(layers) == 0:
        return model(x)

    first_trainable = get_trainable_layer_start(model)

    if first_trainable <= 0:
        h = model.encode_inputs(x)
        for layer in layers:
            h = layer(h)
        return h

    with torch.no_grad():
        h = model.encode_inputs(x)
        for layer in layers[:first_trainable]:
            h = layer(h)

    h = h.detach()

    for layer in layers[first_trainable:]:
        h = layer(h)

    return h


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = hlcm_encode_full_memory_safe(model=model, x=x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        ec = ec.reshape(B, (k1 - k0), -1)
        ec = F.normalize(ec, dim=-1)

        chunk_logits = torch.einsum("bd,bkd->bk", e_q, ec)
        logits_list.append(chunk_logits)

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
    )
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# INTERPRETABLE EVAL METRICS
# ============================================================

@torch.no_grad()
def evaluate_hlcm_detailed(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
    split_name: str = "",
) -> Dict[str, float]:
    if loader is None:
        return {
            "loss": 0.0,
            "acc": 0.0,
            "chance_acc": 0.0,
            "mean_num_choices": 0.0,
            "mean_gold_prob": 0.0,
            "mean_pred_prob": 0.0,
            "mean_gold_rank": 0.0,
            "top2_acc": 0.0,
            "margin_gold_minus_best_neg": 0.0,
        }

    model.eval()

    tot_loss = 0.0
    tot_acc = 0.0
    tot_top2 = 0.0
    tot_gold_prob = 0.0
    tot_pred_prob = 0.0
    tot_gold_rank = 0.0
    tot_margin = 0.0
    tot_chance = 0.0
    tot_num_choices = 0.0
    n = 0

    for batch in loader:
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        labels = batch["label"].to(logits.device, non_blocking=True)
        loss = F.cross_entropy(logits, labels)

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=1)

        bs = labels.size(0)
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        tot_loss += float(loss.item()) * bs
        tot_acc += float((preds == labels).float().sum().item())

        for i in range(bs):
            y = int(labels[i].item())
            p = int(preds[i].item())
            valid_k = int(choice_counts[i])

            probs_i = probs[i, :valid_k]
            logits_i = logits[i, :valid_k]

            gold_prob = float(probs_i[y].item())
            pred_prob = float(probs_i[p].item())

            order = torch.argsort(logits_i, descending=True)
            gold_rank = int((order == y).nonzero(as_tuple=False).view(-1)[0].item()) + 1

            topk = min(2, valid_k)
            top2_hit = int(y in order[:topk].tolist())

            if valid_k > 1:
                neg_mask = torch.ones(valid_k, dtype=torch.bool, device=logits_i.device)
                neg_mask[y] = False
                best_neg = float(logits_i[neg_mask].max().item())
                margin = float(logits_i[y].item() - best_neg)
            else:
                margin = 0.0

            tot_top2 += top2_hit
            tot_gold_prob += gold_prob
            tot_pred_prob += pred_prob
            tot_gold_rank += gold_rank
            tot_margin += margin
            tot_chance += 1.0 / max(1, valid_k)
            tot_num_choices += valid_k

        n += bs

    out = {
        "loss": tot_loss / max(1, n),
        "acc": tot_acc / max(1, n),
        "chance_acc": tot_chance / max(1, n),
        "mean_num_choices": tot_num_choices / max(1, n),
        "mean_gold_prob": tot_gold_prob / max(1, n),
        "mean_pred_prob": tot_pred_prob / max(1, n),
        "mean_gold_rank": tot_gold_rank / max(1, n),
        "top2_acc": tot_top2 / max(1, n),
        "margin_gold_minus_best_neg": tot_margin / max(1, n),
    }
    return out


@torch.no_grad()
def dump_eval_predictions(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
    out_path: str,
):
    model.eval()
    rows = []

    for batch in loader:
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )
        probs = F.softmax(logits, dim=-1)

        pred_idx = logits.argmax(dim=1).detach().cpu().tolist()
        gold_idx = batch["label"].detach().cpu().tolist()
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        for i in range(len(pred_idx)):
            p = pred_idx[i]
            g = gold_idx[i]
            valid_k = int(choice_counts[i])
            logits_i = logits[i, :valid_k].detach().cpu()
            probs_i = probs[i, :valid_k].detach().cpu()
            order = torch.argsort(logits_i, descending=True)
            gold_rank = int((order == g).nonzero(as_tuple=False).view(-1)[0].item()) + 1

            choices = batch["choice_texts"][i]
            rows.append({
                "gold_answer": batch["gold_answers"][i],
                "pred_answer": choices[p],
                "gold_choice_index": g,
                "pred_choice_index": p,
                "correct": int(p == g),
                "gold_rank": gold_rank,
                "gold_prob": float(probs_i[g].item()),
                "pred_prob": float(probs_i[p].item()),
                "choice_texts": choices,
                "choice_logits": [float(x) for x in logits_i.tolist()],
                "choice_probs": [float(x) for x in probs_i.tolist()],
            })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError("No trainable parameters found.")

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: HLCMMAWPSConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    train_eval_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base_train_eval = evaluate_hlcm_detailed(model, train_eval_loader, mu, sigma, cfg, split_name="train")
    base_val_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg, split_name="validation")

    print(
        f"[SFT][BASE] "
        f"train_eval_acc={base_train_eval['acc']:.4f} "
        f"val_acc={base_val_eval['acc']:.4f} "
        f"chance_train={base_train_eval['chance_acc']:.4f} "
        f"chance_val={base_val_eval['chance_acc']:.4f}"
    )
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss_live": "",
        "train_acc_live": "",
        "train_eval_loss": base_train_eval["loss"],
        "train_eval_acc": base_train_eval["acc"],
        "train_eval_chance_acc": base_train_eval["chance_acc"],
        "train_eval_mean_gold_rank": base_train_eval["mean_gold_rank"],
        "train_eval_top2_acc": base_train_eval["top2_acc"],
        "train_eval_mean_gold_prob": base_train_eval["mean_gold_prob"],
        "train_eval_margin_gold_minus_best_neg": base_train_eval["margin_gold_minus_best_neg"],
        "val_loss": base_val_eval["loss"],
        "val_acc": base_val_eval["acc"],
        "val_chance_acc": base_val_eval["chance_acc"],
        "val_mean_gold_rank": base_val_eval["mean_gold_rank"],
        "val_top2_acc": base_val_eval["top2_acc"],
        "val_mean_gold_prob": base_val_eval["mean_gold_prob"],
        "val_margin_gold_minus_best_neg": base_val_eval["margin_gold_minus_best_neg"],
    })

    best_acc = base_val_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss_live = epoch_loss_sum / max(1, epoch_count)
        train_acc_live = epoch_acc_sum / max(1, epoch_count)

        train_eval = evaluate_hlcm_detailed(model, train_eval_loader, mu, sigma, cfg, split_name="train")
        val_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg, split_name="validation")
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss_live": train_loss_live,
            "train_acc_live": train_acc_live,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss_live": train_loss_live,
            "train_acc_live": train_acc_live,
            "train_eval_loss": train_eval["loss"],
            "train_eval_acc": train_eval["acc"],
            "train_eval_chance_acc": train_eval["chance_acc"],
            "train_eval_mean_num_choices": train_eval["mean_num_choices"],
            "train_eval_mean_gold_rank": train_eval["mean_gold_rank"],
            "train_eval_top2_acc": train_eval["top2_acc"],
            "train_eval_mean_gold_prob": train_eval["mean_gold_prob"],
            "train_eval_mean_pred_prob": train_eval["mean_pred_prob"],
            "train_eval_margin_gold_minus_best_neg": train_eval["margin_gold_minus_best_neg"],
            "val_loss": val_eval["loss"],
            "val_acc": val_eval["acc"],
            "val_chance_acc": val_eval["chance_acc"],
            "val_mean_num_choices": val_eval["mean_num_choices"],
            "val_mean_gold_rank": val_eval["mean_gold_rank"],
            "val_top2_acc": val_eval["top2_acc"],
            "val_mean_gold_prob": val_eval["mean_gold_prob"],
            "val_mean_pred_prob": val_eval["mean_pred_prob"],
            "val_margin_gold_minus_best_neg": val_eval["margin_gold_minus_best_neg"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_live_acc={train_acc_live:.4f} "
            f"train_eval_acc={train_eval['acc']:.4f} "
            f"val_acc={val_eval['acc']:.4f} "
            f"chance={val_eval['chance_acc']:.4f} "
            f"val_gold_rank={val_eval['mean_gold_rank']:.4f} "
            f"val_top2={val_eval['top2_acc']:.4f} "
            f"val_gold_prob={val_eval['mean_gold_prob']:.4f} "
            f"val_margin={val_eval['margin_gold_minus_best_neg']:.4f}"
        )

        if val_eval["acc"] > best_acc:
            best_acc = val_eval["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt, sched, scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_val_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    train_eval_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMMAWPSConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base_train_eval = evaluate_hlcm_detailed(model, train_eval_loader, mu, sigma, cfg, split_name="train")
    base_val_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg, split_name="validation")
    print(
        f"[GRPO][BASE] "
        f"train_eval_acc={base_train_eval['acc']:.4f} "
        f"val_acc={base_val_eval['acc']:.4f} "
        f"chance_train={base_train_eval['chance_acc']:.4f} "
        f"chance_val={base_val_eval['chance_acc']:.4f}"
    )
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss_live": "",
        "train_acc_live": "",
        "train_eval_loss": base_train_eval["loss"],
        "train_eval_acc": base_train_eval["acc"],
        "train_eval_chance_acc": base_train_eval["chance_acc"],
        "train_eval_mean_gold_rank": base_train_eval["mean_gold_rank"],
        "train_eval_top2_acc": base_train_eval["top2_acc"],
        "train_eval_mean_gold_prob": base_train_eval["mean_gold_prob"],
        "train_eval_margin_gold_minus_best_neg": base_train_eval["margin_gold_minus_best_neg"],
        "val_loss": base_val_eval["loss"],
        "val_acc": base_val_eval["acc"],
        "val_chance_acc": base_val_eval["chance_acc"],
        "val_mean_gold_rank": base_val_eval["mean_gold_rank"],
        "val_top2_acc": base_val_eval["top2_acc"],
        "val_mean_gold_prob": base_val_eval["mean_gold_prob"],
        "val_margin_gold_minus_best_neg": base_val_eval["margin_gold_minus_best_neg"],
    })

    best_acc = base_val_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss_live = epoch_loss_sum / max(1, epoch_count)
        train_acc_live = epoch_acc_sum / max(1, epoch_count)

        train_eval = evaluate_hlcm_detailed(model, train_eval_loader, mu, sigma, cfg, split_name="train")
        val_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg, split_name="validation")
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss_live": train_loss_live,
            "train_acc_live": train_acc_live,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss_live": train_loss_live,
            "train_acc_live": train_acc_live,
            "train_eval_loss": train_eval["loss"],
            "train_eval_acc": train_eval["acc"],
            "train_eval_chance_acc": train_eval["chance_acc"],
            "train_eval_mean_num_choices": train_eval["mean_num_choices"],
            "train_eval_mean_gold_rank": train_eval["mean_gold_rank"],
            "train_eval_top2_acc": train_eval["top2_acc"],
            "train_eval_mean_gold_prob": train_eval["mean_gold_prob"],
            "train_eval_mean_pred_prob": train_eval["mean_pred_prob"],
            "train_eval_margin_gold_minus_best_neg": train_eval["margin_gold_minus_best_neg"],
            "val_loss": val_eval["loss"],
            "val_acc": val_eval["acc"],
            "val_chance_acc": val_eval["chance_acc"],
            "val_mean_num_choices": val_eval["mean_num_choices"],
            "val_mean_gold_rank": val_eval["mean_gold_rank"],
            "val_top2_acc": val_eval["top2_acc"],
            "val_mean_gold_prob": val_eval["mean_gold_prob"],
            "val_mean_pred_prob": val_eval["mean_pred_prob"],
            "val_margin_gold_minus_best_neg": val_eval["margin_gold_minus_best_neg"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_live_acc={train_acc_live:.4f} "
            f"train_eval_acc={train_eval['acc']:.4f} "
            f"val_acc={val_eval['acc']:.4f} "
            f"chance={val_eval['chance_acc']:.4f} "
            f"val_gold_rank={val_eval['mean_gold_rank']:.4f} "
            f"val_top2={val_eval['top2_acc']:.4f} "
            f"val_gold_prob={val_eval['mean_gold_prob']:.4f} "
            f"val_margin={val_eval['margin_gold_minus_best_neg']:.4f}"
        )

        if val_eval["acc"] > best_acc:
            best_acc = val_eval["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt, sched, scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_val_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_mawps_hybrid_hlcm(dataset_name: str, cfg: HLCMMAWPSConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, _ = load_mawps_local(cfg)
    answer_pool = collect_answer_pool(train_hf)
    print(f"[mawps] unique training answers in pool: {len(answer_pool)}")

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer, answer_pool)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer, answer_pool)

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    train_eval_loader = DataLoader(
        train_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError("No trainable parameters found after applying finetune mode.")

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    # exact chance baselines from cached splits
    train_true_chance = sum(1.0 / max(1, r["num_choices"]) for r in train_rows) / max(1, len(train_rows))
    val_true_chance = sum(1.0 / max(1, r["num_choices"]) for r in eval_rows) / max(1, len(eval_rows))
    print(f"[chance] train_true_chance={train_true_chance:.4f} val_true_chance={val_true_chance:.4f}")

    metadata_base = {
        "dataset": dataset_name,
        "dataset_type": "mawps_answer_ranking",
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_choices": cfg.num_choices,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
        "train_true_chance_acc": train_true_chance,
        "val_true_chance_acc": val_true_chance,
    }

    print(f"\n========== {dataset_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
    }

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        train_eval_loader=train_eval_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    print(f"\n========== {dataset_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        train_eval_loader=train_eval_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_train_eval = evaluate_hlcm_detailed(hlcm, train_eval_loader, mu, sigma, cfg, split_name="train")
    final_val_eval = evaluate_hlcm_detailed(hlcm, eval_loader, mu, sigma, cfg, split_name="validation")
    mem = gpu_mem_mb(device)

    dump_eval_predictions(
        model=hlcm,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        out_path=os.path.join(out_dir, "eval_predictions.json"),
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_train_eval": final_train_eval,
            "final_val_eval": final_val_eval,
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "dataset_type": "mawps_answer_ranking",
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],

        "final_train_eval_loss": final_train_eval["loss"],
        "final_train_eval_acc": final_train_eval["acc"],
        "final_train_eval_chance_acc": final_train_eval["chance_acc"],
        "final_train_eval_mean_gold_rank": final_train_eval["mean_gold_rank"],
        "final_train_eval_top2_acc": final_train_eval["top2_acc"],
        "final_train_eval_mean_gold_prob": final_train_eval["mean_gold_prob"],
        "final_train_eval_margin_gold_minus_best_neg": final_train_eval["margin_gold_minus_best_neg"],

        "final_val_loss": final_val_eval["loss"],
        "final_val_acc": final_val_eval["acc"],
        "final_val_chance_acc": final_val_eval["chance_acc"],
        "final_val_mean_gold_rank": final_val_eval["mean_gold_rank"],
        "final_val_top2_acc": final_val_eval["top2_acc"],
        "final_val_mean_gold_prob": final_val_eval["mean_gold_prob"],
        "final_val_margin_gold_minus_best_neg": final_val_eval["margin_gold_minus_best_neg"],

        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "choice_chunk_size": cfg.choice_chunk_size,
        "num_choices": cfg.num_choices,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "ref_logits_path": ref_logits_path,
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
        "train_true_chance_acc": train_true_chance,
        "val_true_chance_acc": val_true_chance,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[HYBRID][FINAL] dataset={dataset_name} "
        f"train_eval_acc={final_train_eval['acc']:.4f} "
        f"val_acc={final_val_eval['acc']:.4f} "
        f"val_chance={final_val_eval['chance_acc']:.4f} "
        f"val_gold_rank={final_val_eval['mean_gold_rank']:.4f} "
        f"val_top2={final_val_eval['top2_acc']:.4f} "
        f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = HLCMMAWPSConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, entropy_bonus={cfg.entropy_bonus}"
    )
    print(f"num_choices={cfg.num_choices}, choice_chunk_size={cfg.choice_chunk_size}")

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_mawps_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=1, bs_eval=1, grad_accum=8
SFT lr=5e-05, GRPO lr=1e-05, GRPO group_size=2, beta_kl=0.02, entropy_bonus=0.001
num_choices=4, choice_chunk_size=1

==================== mawps_local ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[mawps] unique training answers in pool: 359
[cache] building mawps_local / train


cache:mawps_local:train: 100%|██████████████████████████████████| 1417/1417 [02:33<00:00,  9.25it/s]


...

[SFT][epoch 2/2] train_live_acc=0.2915 train_eval_acc=0.5533 val_acc=0.5352 chance=0.2500 val_gold_rank=1.7690 val_top2=0.7746 val_gold_prob=0.2503 val_margin=0.0003
[SFT][FINAL] best_val_acc=0.5352 total_train_time=53.48 min
[ref_logits] building mawps_cached_ref_logits/mawps_local_train_ref_logits_tok256_seq8_K4.pt


precompute_ref_logits: 100%|████████████████████████████████████| 1417/1417 [05:49<00:00,  4.05it/s]


[ref_logits] saved mawps_cached_ref_logits/mawps_local_train_ref_logits_tok256_seq8_K4.pt (1417 rows)

========== mawps_local :: STAGE 2 / GRPO ==========
[GRPO][BASE] train_eval_acc=0.5526 val_acc=0.5352 chance_train=0.2500 chance_val=0.2500


grpo epoch 1/1: 100%|█| 1417/1417 [20:26<00:00,  1.15it/s, acc=0.2999, kl=0.0134, loss=-0.0030, lr=0


[GRPO][epoch 1/1] train_live_acc=0.2999 train_eval_acc=0.5526 val_acc=0.5352 chance=0.2500 val_gold_rank=1.7690 val_top2=0.7746 val_gold_prob=0.2504 val_margin=0.0004
[GRPO][FINAL] best_val_acc=0.5352 total_train_time=28.04 min
[HYBRID][FINAL] dataset=mawps_local train_eval_acc=0.5526 val_acc=0.5352 val_chance=0.2500 val_gold_rank=1.7690 val_top2=0.7746 max_gpu_alloc=34190.7 MB saved -> runs/hlcm_mawps_local_interpretable/mawps_local

All done. Outputs in: runs/hlcm_mawps_local_interpretable
Total wall time: 117.32 min
